# CALCUALTE GENE-SET BURDEN SCORES 

# GDC SOMATIC MUTATIONS 
This script calculates the rare variant polygenic risk score for gene sets derived from GDC somatic mutations. The gene-level polygenic risk score files are first downloaded from the workspace bucket. Each gene set is then processed in a loop. For each gene in a given set, the script identifies the corresponding gene-level polygenic risk score file and adds its value to a running total. The final sum represents the polygenic risk score for the entire gene set.

In [5]:
# Set bucket environment
# Load packages
import os
import re
import os
import gc
import glob
import pandas as pd 
from biomart import BiomartServer
from functools import reduce

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261029 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

In [6]:
import os
import glob
import subprocess

import pandas as pd
import numpy as np
import statsmodels.api as sm


# =============================================================================================================
# Set up directories
# =============================================================================================================

OUTPUT_DIR = "burden_polygenic_score"
SUMMARY_DIR = "logistic_summary"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]

GOOGLE_CLOUD_PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]

# =============================================================================================================
# Copy files from Google Cloud Storage
# =============================================================================================================

# Copy gene-set burden score files
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_gene_sets/somatic_mutations/*remove*PRS.tsv {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy phenotype file
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/saige_phenotype_files/aml_phenotype_files/EUR_phenotype_file.txt {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy exome burden file
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/gene_burden_analysis/burden_gene_sets/exome/* {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy genomic metrics file
subprocess.run(
    f"cp /home/dataproc/workspace/vwb-aou-datasets-controlled-v8r9/v8/wgs/short_read/snpindel/aux/qc/genomics_metrics_Dec142023_1859_02_tz0000.tsv {OUTPUT_DIR}/",
    shell=True,
    check=True
)

# Copy pca file
subprocess.run(
    f"gsutil cp {WORKSPACE_BUCKET}/data/pcs/EUR_flashpca.pcs {OUTPUT_DIR}/",
    shell=True,
    check=True
)


# =============================================================================================================
# List gene-set burden score files
# =============================================================================================================

files = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "*remove*PRS.tsv"))
)

print(f"Found {len(files)} gene-set burden score files.")


# =============================================================================================================
# Load files necessary for logistic regression
# =============================================================================================================

# Load phenotype file
pheno = pd.read_csv(
    os.path.join(OUTPUT_DIR, "EUR_phenotype_file.txt"),
    sep="\t"
)

# Load exome burden file
exome = pd.read_csv(
    os.path.join(OUTPUT_DIR, "exome-burden-PRS.tsv"),
    sep="\t"
)

# Load genomic metric file
metrics = pd.read_csv(
    os.path.join(OUTPUT_DIR, "genomics_metrics_Dec142023_1859_02_tz0000.tsv"),
    sep="\t"
)


# Format exome burden file
exome = (
    exome.rename(columns={
        "IID": "person_id",
        "EXOME_SCORE1_SUM": "exome_burden_prs"
    })
    [["person_id", "exome_burden_prs"]]
)

# Format genomic metric file
metrics = (
    metrics.rename(columns={
        "research_id": "person_id"
    })
    [["person_id", "sample_source"]]
)

eur_flashpca = pd.read_csv(
    "burden_polygenic_score/EUR_flashpca.pcs",
    sep="\t"
)[["IID", "group"]].rename(
    columns={"IID": "person_id"}
)

eur_flashpca = eur_flashpca[
    eur_flashpca["group"] == "Unrelated_Inliers"
][['person_id']].copy()

# =============================================================================================================
# Logistic regression of gene sets for exome burden polygenic risk score
# =============================================================================================================

covariates = [
    "sex", "age",
    "PC1", "PC2", "PC3", "PC4", "PC5",
    "PC6", "PC7", "PC8", "PC9", "PC10"
]

for file in files:

    gene_set = os.path.basename(file).replace("-PRS.tsv", "")

    print(f"\nProcessing file: {gene_set} ...")

    # Load gene-set burden score
    score = pd.read_csv(file, sep="\t")

    # Format gene-set burden score
    score = (
        score.rename(columns={
            "IID": "person_id",
            "TOTAL_SCORE1_SUM": "burden_prs"
        })
        [["person_id", "burden_prs"]]
    )

    # Merge all data frames
    merged_df = (
        pheno
        .merge(score, on="person_id", how="inner")
        .merge(exome, on="person_id", how="inner")
        .merge(metrics, on="person_id", how="inner")
        .merge(eur_flashpca, on="person_id", how="inner")
    )

    print(f"Number of individuals: {len(merged_df)}")

    # Skip gene sets with no rare variants
    if merged_df["burden_prs"].max() == 0:

        print(f"Skipping file: {gene_set} (no rare variants)")

        continue

    # Standardize burden scores
    merged_df["burden_prs"] = (
        merged_df["burden_prs"] - merged_df["burden_prs"].mean()
    ) / merged_df["burden_prs"].std()

    merged_df["exome_burden_prs"] = (
        merged_df["exome_burden_prs"] - merged_df["exome_burden_prs"].mean()
    ) / merged_df["exome_burden_prs"].std()

    # Define outcome and predictors
    X = merged_df[
        ["burden_prs", "exome_burden_prs"] + covariates
    ]

    y = merged_df["aml_phenotype"]

    # Add intercept
    X = sm.add_constant(X)

    # Run logistic regression
    logistic = sm.Logit(y, X).fit(disp=False)

    # Extract regression coefficients and statistics
    logistic_summary = pd.DataFrame({
        "Coefficient": logistic.params,
        "Std_Error": logistic.bse,
        "z_value": logistic.tvalues,
        "p_value": logistic.pvalues,
        "OR": np.exp(logistic.params),
        "CI_lower": np.exp(logistic.conf_int()[0]),
        "CI_upper": np.exp(logistic.conf_int()[1])
    })
    
    print(logistic_summary)

    # Save summary
    output_file = os.path.join(
        SUMMARY_DIR,
        f"{gene_set}-unrelated-logistic-summary.tsv"
    )

    logistic_summary.to_csv(
        output_file,
        sep="\t",
        index=True,
        index_label="Variable"
    )

    print(f"Saved results: {output_file}")
    

# Copy exome burden file
subprocess.run(
    f"gsutil cp {OUTPUT_DIR}/* {WORKSPACE_BUCKET}/data/gene_burden_analysis/results/",
    shell=True,
    check=True
)

print("\nFinished processing all gene sets.")

Copying gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_gene_sets/somatic_mutations/aml-frequently-mutated-census-genes-protein-coding-remove-chip-PRS.tsv...
Copying gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_gene_sets/somatic_mutations/aml-frequently-mutated-census-genes-protein-coding-vep-high-remove-chip-PRS.tsv...
Copying gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_gene_sets/somatic_mutations/aml-frequently-mutated-census-genes-protein-coding-vep-moderate-high-remove-chip-PRS.tsv...
Copying gs://rw-migration-aou-rw-9f879fc4/data/gene_burden_analysis/burden_gene_sets/somatic_mutations/cancer-frequently-mutated-census-genes-protein-coding-remove-chip-PRS.tsv...
| [4 files][ 10.0 MiB/ 10.0 MiB]                                                
==> NOTE: You are performing a sequence of gsutil operations that may
run significantly faster if you instead use gsutil -m cp ... Please
see the -m section under "gsutil help optio

Found 6 gene-set burden score files.

Processing file: aml-frequently-mutated-census-genes-protein-coding-remove-chip ...
Number of individuals: 151774
                  Coefficient  Std_Error    z_value       p_value        OR  \
const               -6.547240   0.106217 -61.640261  0.000000e+00  0.001434   
burden_prs           0.095293   0.063111   1.509927  1.310620e-01  1.099981   
exome_burden_prs     0.066572   0.142498   0.467180  6.403708e-01  1.068838   
sex                 -0.087907   0.132947  -0.661221  5.084704e-01  0.915846   
age                  0.401989   0.074371   5.405194  6.473827e-08  1.494795   
PC1                  0.017309   0.138346   0.125117  9.004308e-01  1.017460   
PC2                 -0.127680   0.074235  -1.719940  8.544336e-02  0.880135   
PC3                  0.061448   0.056678   1.084152  2.782972e-01  1.063375   
PC4                 -0.029909   0.062425  -0.479114  6.318575e-01  0.970534   
PC5                 -0.048729   0.069838  -0.697747  4.853

Copying file://burden_polygenic_score/EUR_flashpca.pcs [Content-Type=application/octet-stream]...
Copying file://burden_polygenic_score/EUR_phenotype_file.txt [Content-Type=text/plain]...
Copying file://burden_polygenic_score/aml-frequently-mutated-census-genes-protein-coding-remove-chip-PRS.tsv [Content-Type=text/tab-separated-values]...
Copying file://burden_polygenic_score/aml-frequently-mutated-census-genes-protein-coding-vep-high-remove-chip-PRS.tsv [Content-Type=text/tab-separated-values]...
/ [4 files][106.4 MiB/106.4 MiB]                                                
==> NOTE: You are performing a sequence of gsutil operations that may
run significantly faster if you instead use gsutil -m cp ... Please
see the -m section under "gsutil help options" for further information
about when gsutil -m can be advantageous.

Copying file://burden_polygenic_score/aml-frequently-mutated-census-genes-protein-coding-vep-moderate-high-remove-chip-PRS.tsv [Content-Type=text/tab-separated-valu


Finished processing all gene sets.
